# Play-by-Play Database Schema Proposal

**Issue #140** — Design artifact for the NHL Dashboard play-by-play pipeline.

This notebook explores the `/v1/gamecenter/{game_id}/play-by-play` API response and evaluates three
schema options for storing its events in SQLite. Run all cells top-to-bottom against a real API
response, then record the chosen approach in **Section 7 — Recommendation** before handing off
to implementation.

**Do not write SQLAlchemy models or persist functions until Section 7 is filled in.**

---

## Setup

```bash
pip install jupyter pandas httpx
jupyter notebook nhl-dashboard/notebooks/play_by_play_schema_proposal.ipynb
```

Set `GAME_ID` below to any completed NHL game. Playoff game IDs follow the pattern
`YYYY030RRGG` where YYYY = season start year, RR = round (01–04), GG = game in series.

In [ ]:
import json
from collections import defaultdict
from pprint import pprint

import httpx
import pandas as pd

# Change to any completed game_id — e.g., a 2025-26 playoff game.
# Format: YYYY03RRGG (season year, playoffs=03, round, game-in-series).
GAME_ID = 2025030411
NHL_BASE = "https://api-web.nhle.com/v1"

resp = httpx.get(f"{NHL_BASE}/gamecenter/{GAME_ID}/play-by-play", timeout=15)
resp.raise_for_status()
pbp = resp.json()

plays = pbp.get("plays", [])

away = pbp.get("awayTeam", {})
home = pbp.get("homeTeam", {})
print(f"Game ID    : {pbp.get('id')}")
print(f"Game date  : {pbp.get('gameDate')}")
print(f"Match-up   : {away.get('abbrev')} @ {home.get('abbrev')}")
print(f"Total plays: {len(plays)}")
print(f"Top-level keys: {list(pbp.keys())}")

## Section 1 — API Exploration

Call `/v1/gamecenter/{game_id}/play-by-play` for a known completed game, pretty-print the raw
JSON shape, and display key top-level fields.

Key top-level fields in the response:

| Field | Description |
|---|---|
| `id` | NHL gamePk |
| `gameDate` | Date string `YYYY-MM-DD` |
| `awayTeam` / `homeTeam` | Team identity objects |
| `plays` | Array of all play events |
| `rosterSpots` | Player roster with jersey numbers and positions |
| `gameVideo` | Optional video clip references |

In [ ]:
print(f"Total play events: {len(plays)}")
print()
print("--- Raw JSON shape: first 3 plays ---")
for play in plays[:3]:
    pprint(play)
    print()

## Section 2 — Event Classification

The `plays[]` array contains two fundamentally different event categories:

### Player-action events
Events tied to a specific player on the ice — each carry player references, ice coordinates
(x/y), zone code, period, and time elapsed. Types include:
`goal`, `hit`, `shot-on-goal`, `blocked-shot`, `penalty`, `faceoff`, `missed-shot`,
`giveaway`, `takeaway`, `delayed-penalty`.

### Game-state events
Administrative / structural events that mark game transitions — no player or coordinate data.
Types include: `period-start`, `period-end`, `game-start`, `game-end`, `intermission`,
`stoppage`.

This distinction is the core architectural driver: the two categories share almost no
columns, which makes a single flat table very wide with lots of NULLs.

In [ ]:
# Group plays by typeDescKey to see all event types and counts.
type_groups: dict[str, list] = defaultdict(list)
for play in plays:
    key = play.get("typeDescKey", "unknown")
    type_groups[key].append(play)

print("--- Event type counts (descending) ---")
for tdk, group in sorted(type_groups.items(), key=lambda x: -len(x[1])):
    print(f"  {tdk:<30} {len(group)}")

# Classify into the two categories.
PLAYER_ACTION_TYPES = {
    "goal", "hit", "shot-on-goal", "blocked-shot", "penalty",
    "faceoff", "missed-shot", "giveaway", "takeaway", "delayed-penalty",
}
GAME_STATE_TYPES = {
    "period-start", "period-end", "game-start", "game-end",
    "intermission", "stoppage",
}

player_events = [p for p in plays if p.get("typeDescKey") in PLAYER_ACTION_TYPES]
game_state_events = [p for p in plays if p.get("typeDescKey") in GAME_STATE_TYPES]
other_events = [
    p for p in plays
    if p.get("typeDescKey") not in PLAYER_ACTION_TYPES | GAME_STATE_TYPES
]

print()
print(f"Player-action events : {len(player_events)}")
print(f"Game-state events    : {len(game_state_events)}")
print(f"Other / unclassified : {len(other_events)}")
if other_events:
    other_keys = {p.get("typeDescKey") for p in other_events}
    print(f"  Unclassified keys  : {other_keys}")

# Show a representative sample from each category.
if player_events:
    print()
    print("--- Sample player-action event ---")
    pprint(player_events[0])

if game_state_events:
    print()
    print("--- Sample game-state event ---")
    pprint(game_state_events[0])

## Section 3 — Field Inventory

Tabulate which top-level fields appear on which event types (player refs, coordinates,
time, period, etc.) to expose nullable vs. required columns.

A `✓` means the field is present on at least one event of that type.
Blanks indicate the field is always absent — a strong signal that the column
would be nullable in any merged-table design.

In [ ]:
def top_level_fields(play: dict) -> set:
    """Return the set of top-level keys present on a play event."""
    return set(play.keys())


# Collect fields per typeDescKey.
field_by_type: dict[str, set] = defaultdict(set)
for play in plays:
    tdk = play.get("typeDescKey", "unknown")
    field_by_type[tdk] |= top_level_fields(play)

all_fields = sorted({f for fields in field_by_type.values() for f in fields})
ordered_types = sorted(field_by_type.keys())

rows = []
for field in all_fields:
    row = {"field": field}
    for tdk in ordered_types:
        row[tdk] = "\u2713" if field in field_by_type[tdk] else ""
    rows.append(row)

df = pd.DataFrame(rows).set_index("field")
print("Field presence by event type (\u2713 = present):")
display(df)

# Highlight fields that only appear on player-action events (never on game-state events).
player_only = [
    f for f in all_fields
    if any(f in field_by_type[t] for t in PLAYER_ACTION_TYPES if t in field_by_type)
    and not any(f in field_by_type[t] for t in GAME_STATE_TYPES if t in field_by_type)
]
print()
print("Fields present ONLY on player-action events (always NULL for game-state rows):")
print(player_only)

## Section 4 — Schema Option A: Single Table

One `play_event` table stores all event types. Columns that don't apply to a given
event type are left NULL.

```sql
CREATE TABLE play_event (
    id              INTEGER  PRIMARY KEY AUTOINCREMENT,
    game_id         INTEGER  NOT NULL,   -- FK → game.game_id
    event_idx       INTEGER  NOT NULL,
    period_num      INTEGER,
    period_type     TEXT,                -- 'REG', 'OT', 'SO'
    time_in_period  TEXT,                -- 'MM:SS' elapsed
    time_remaining  TEXT,                -- 'MM:SS' remaining
    type_code       INTEGER  NOT NULL,
    type_desc_key   TEXT     NOT NULL,
    sort_order      INTEGER,
    -- coordinate fields (NULL for game-state events)
    x_coord         REAL,
    y_coord         REAL,
    zone_code       TEXT,                -- 'O', 'N', 'D'
    -- player references (NULL for game-state events)
    player1_id      INTEGER,
    player1_role    TEXT,                -- 'scorer', 'hitter', 'shooter', etc.
    player2_id      INTEGER,
    player2_role    TEXT,
    player3_id      INTEGER,
    player3_role    TEXT,
    -- goal-specific (NULL for non-goal events)
    shot_type       TEXT,
    goal_modifier   TEXT,                -- 'none', 'pp', 'sh', 'en'
    away_score      INTEGER,
    home_score      INTEGER,
    -- penalty-specific (NULL for non-penalty events)
    penalty_type    TEXT,
    penalty_mins    INTEGER,
    UNIQUE (game_id, event_idx)
);
```

### Trade-off analysis

| | Detail |
|---|---|
| **Pros** | Single JOIN for any query. Simplest ORM model. Easy to add a new event type without a migration. Straightforward upsert logic. |
| **Cons** | Very wide table — ~50–60% of columns are NULL for game-state events. Cannot enforce NOT NULL on player-specific columns at the DB layer. Hard to add a CHECK constraint per event type. Schema grows unbounded as new event types add columns. |

## Section 5 — Schema Option B: Base + Child Tables

A lean `play_event` base table holds common fields for every event. Typed child tables
(`goal_event`, `hit_event`, etc.) store event-specific columns via a 1-to-1 FK.

```sql
-- Base table: one row per event regardless of type.
CREATE TABLE play_event (
    id              INTEGER  PRIMARY KEY AUTOINCREMENT,
    game_id         INTEGER  NOT NULL,   -- FK → game.game_id
    event_idx       INTEGER  NOT NULL,
    period_num      INTEGER,
    period_type     TEXT,
    time_in_period  TEXT,
    time_remaining  TEXT,
    type_code       INTEGER  NOT NULL,
    type_desc_key   TEXT     NOT NULL,
    sort_order      INTEGER,
    UNIQUE (game_id, event_idx)
);

-- Child table for goal events.
CREATE TABLE goal_event (
    play_event_id   INTEGER  PRIMARY KEY REFERENCES play_event(id),
    scorer_id       INTEGER  NOT NULL,
    assist1_id      INTEGER,
    assist2_id      INTEGER,
    shot_type       TEXT,
    goal_modifier   TEXT,
    away_score      INTEGER  NOT NULL,
    home_score      INTEGER  NOT NULL,
    x_coord         REAL,
    y_coord         REAL,
    zone_code       TEXT
);

-- Child table for shot-on-goal and missed-shot events.
CREATE TABLE shot_event (
    play_event_id   INTEGER  PRIMARY KEY REFERENCES play_event(id),
    shooter_id      INTEGER  NOT NULL,
    goalie_id       INTEGER,
    shot_type       TEXT,
    x_coord         REAL,
    y_coord         REAL,
    zone_code       TEXT
);

-- Child table for hit events.
CREATE TABLE hit_event (
    play_event_id   INTEGER  PRIMARY KEY REFERENCES play_event(id),
    hitter_id       INTEGER  NOT NULL,
    hittee_id       INTEGER,
    x_coord         REAL,
    y_coord         REAL,
    zone_code       TEXT
);

-- Child table for penalty events.
CREATE TABLE penalty_event (
    play_event_id   INTEGER  PRIMARY KEY REFERENCES play_event(id),
    player_id       INTEGER,
    penalty_type    TEXT     NOT NULL,
    duration_mins   INTEGER  NOT NULL,
    team_abbrev     TEXT
);

-- Child table for blocked-shot events.
CREATE TABLE blocked_shot_event (
    play_event_id   INTEGER  PRIMARY KEY REFERENCES play_event(id),
    blocker_id      INTEGER,
    shooter_id      INTEGER,
    x_coord         REAL,
    y_coord         REAL,
    zone_code       TEXT
);

-- Child table for faceoff events.
CREATE TABLE faceoff_event (
    play_event_id   INTEGER  PRIMARY KEY REFERENCES play_event(id),
    winner_id       INTEGER,
    loser_id        INTEGER,
    x_coord         REAL,
    y_coord         REAL,
    zone_code       TEXT
);
```

### Trade-off analysis

| | Detail |
|---|---|
| **Pros** | Clean normalization — each child table enforces NOT NULL on its own columns. Easy to add per-type constraints or indexes. Base table stays lean as new event types arrive. |
| **Cons** | Every typed query requires a JOIN (base + child). Upsert logic is more complex (must insert base row first, then child). ORM models are more verbose — one class per event type. If a new event type doesn't fit an existing child, you add a whole new table. |

## Section 6 — Schema Option C: Two-Table Split

Two tables aligned with the natural event categories: `player_event` for all
player-action events, and `game_state_event` for all structural/administrative events.

```sql
-- Player-action events: goals, hits, shots, blocked shots, penalties, faceoffs.
CREATE TABLE player_event (
    id              INTEGER  PRIMARY KEY AUTOINCREMENT,
    game_id         INTEGER  NOT NULL,   -- FK → game.game_id
    event_idx       INTEGER  NOT NULL,
    period_num      INTEGER  NOT NULL,
    period_type     TEXT,
    time_in_period  TEXT     NOT NULL,
    time_remaining  TEXT,
    type_code       INTEGER  NOT NULL,
    type_desc_key   TEXT     NOT NULL,   -- 'goal','hit','shot-on-goal',
                                         -- 'blocked-shot','penalty','faceoff', ...
    sort_order      INTEGER,
    x_coord         REAL,
    y_coord         REAL,
    zone_code       TEXT,
    player1_id      INTEGER,
    player1_role    TEXT,
    player2_id      INTEGER,
    player2_role    TEXT,
    player3_id      INTEGER,
    player3_role    TEXT,
    -- goal-specific nullable columns
    shot_type       TEXT,
    goal_modifier   TEXT,
    away_score      INTEGER,
    home_score      INTEGER,
    -- penalty-specific nullable columns
    penalty_type    TEXT,
    penalty_mins    INTEGER,
    UNIQUE (game_id, event_idx)
);

-- Game-state events: period-start, period-end, intermission, game-start, game-end.
CREATE TABLE game_state_event (
    id              INTEGER  PRIMARY KEY AUTOINCREMENT,
    game_id         INTEGER  NOT NULL,   -- FK → game.game_id
    event_idx       INTEGER  NOT NULL,
    period_num      INTEGER,
    period_type     TEXT,
    time_in_period  TEXT,
    time_remaining  TEXT,
    type_code       INTEGER  NOT NULL,
    type_desc_key   TEXT     NOT NULL,   -- 'period-start','period-end',
                                         -- 'intermission','game-start','game-end'
    UNIQUE (game_id, event_idx)
);
```

### Trade-off analysis

| | Detail |
|---|---|
| **Pros** | Directly mirrors the two natural event categories. Queries within a single category need no JOIN. `game_state_event` stays very narrow. Simple upsert — route each play to the right table. Two ORM models covers the entire domain. |
| **Cons** | `player_event` still has nullable goal/penalty columns — same issue as Option A but scoped to one table. Cross-category queries (e.g., "all events in period 2") require a UNION. Sub-type constraints (NOT NULL on scorer_id for goals only) still can't be enforced at the DB layer. |

## Section 7 — Recommendation

Record the chosen schema approach and rationale here before handing off to implementation.
This decision becomes the implementation brief for the SQLAlchemy models and persist functions.

---

**Chosen approach:** *(developer fills in — Option A / B / C)*

**Rationale:** *(briefly describe why this option fits the current query patterns and
acceptable trade-offs for the NHL Dashboard use case)*

**Primary query pattern this schema must support:**
*(e.g., "fetch all goals for a game_id to compute expected goals", "stream events
by period for a live game view", etc.)*

**Accepted trade-offs:**
*(note which cons from the chosen option you are explicitly accepting)*

---

*Once this section is complete, open a new GitHub issue referencing this notebook
and requesting SQLAlchemy model(s) + persist function(s) for the chosen schema.*